# Bank Marketing — analiza eksploracyjna i modele bazowe
Analiza danych UCI, walidacja wejścia i porównanie klasyfikatorów. Logika eksperymentu jest wspólna z CLI.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
if not (root / "pyproject.toml").exists():
    raise RuntimeError("Uruchom notebook w katalogu repo lub notebooks.")
sys.path.insert(0, str(root))

## Środowisko
Import konfiguracji i funkcji eksperymentu z katalogu projektu.

In [ ]:
import pandas as pd

from src.config import load_config
from src.data.validation import validate_dataframe
from src.pipeline import evaluate, prepare, train

config = load_config()

## Integralność i charakterystyka danych
prepare sprawdza SHA-256, waliduje strukturę i tworzy chronologiczne części. Statystyki pełnego CSV opisują zbiór; nie są wynikiem ewaluacji końcowego testu.

In [ ]:
data_report = prepare(config)
df = pd.read_csv(root / config["data"]["raw"], sep=";")
print(df.shape)
print(df["y"].value_counts())
display(df.head())
display(df.dtypes)
display(df.isna().sum())
print("Duplikaty:", df.duplicated().sum())
print("unknown:", data_report["unknown_counts"])
print("pdays=999:", data_report["pdays_999"])

## Wartości specjalne
unknown oznacza brak informacji, nonexistent brak wyniku poprzedniej kampanii, a pdays=999 brak poprzedniego kontaktu. Identyczne wiersze nie są automatycznie usuwane.

## Walidacja negatywna
Kopia bez targetu musi zostać odrzucona przed treningiem.

In [ ]:
try:
    validate_dataframe(df.drop(columns="y"))
except ValueError as error:
    print("Oczekiwane odrzucenie:", error)
else:
    raise AssertionError("Walidator zaakceptował błędne dane")

## Rozkład klas i podział
Podział 60/15/10/15 zachowuje kolejność źródłową. Fit jest ograniczony do train. Etykieta, duration oraz niepotwierdzone cechy bieżącego kontaktu nie są wejściami modelu.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(root / "artifacts/reports/class_distribution.png")))
print(data_report["splits"])
print("Cechy:", config["data"]["features"])

## Porównanie modeli
Dummy przewiduje klasę większościową. LogisticRegression korzysta ze skalowania liczb i kodowania kategorii we wspólnym Pipeline. Metryki są liczone na validation.

In [ ]:
training = train(config)
metrics = evaluate(config)
display(pd.DataFrame(metrics["models"]).T)
for name in ["dummy", "model"]:
    display(Image(filename=str(root / f"artifacts/reports/confusion_matrix_{name}.png")))

## Ograniczenia
AP opisuje ranking, a macro F1 uwzględnia obie klasy. LogisticRegression przy progu 0,5 wykrywa 7 z 794 przypadków yes mimo poprawy AP. Final test i replay nie są oceniane. Dane historyczne, brak identyfikatorów osób i brak pełnych snapshotów ograniczają interpretację wyniku.